# HybridChain — IoT Telemetry Anomaly Model (Kaggle trainer)

Trains a compact **autoencoder** (and an Isolation Forest baseline) on IoT sensor telemetry, then exports:

1. `iot_autoencoder.onnx` — for ONNX Runtime (Java) if you want native inference.
2. `iot_anomaly_model.json` — scaler + layer weights + threshold, so the Java node can do inference in **pure Java** with no native dependency (matches this project's lightweight, edge-friendly design).

It runs with **no dataset attached** (it synthesizes realistic multi-sensor telemetry). To use real data, attach a Kaggle IoT dataset and point `CSV_PATH`/`FEATURES` at it.

Runtime: CPU is fine. No GPU needed.

In [ ]:
import subprocess, sys
def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
# tf2onnx/onnx only needed for the .onnx export; JSON export needs none of them.
pip('tf2onnx', 'onnx', 'onnxruntime')

import os, json, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support

np.random.seed(42); tf.random.set_seed(42)
OUT = '/kaggle/working'; os.makedirs(OUT, exist_ok=True)
print('TF', tf.__version__)

In [ ]:
# ---- Data: real CSV if provided, else synthesize ----
CSV_PATH = None          # e.g. '/kaggle/input/your-dataset/telemetry.csv'
FEATURES = None          # e.g. ['temp','humidity','vibration','voltage']
LABEL_COL = None         # optional 0/1 anomaly column for evaluation

def synthesize(n=20000, d=6, anomaly_frac=0.05):
    n_norm = int(n * (1 - anomaly_frac))
    normal = np.random.normal(0, 1, size=(n_norm, d))
    t = np.linspace(0, 50, n_norm)
    normal[:, 0] += np.sin(t)                 # e.g. daily temperature cycle
    normal[:, 1] += 0.5 * normal[:, 0]        # correlated sensors
    n_anom = n - n_norm
    anom = np.random.normal(0, 1, size=(n_anom, d))
    hit = np.random.randint(0, d, n_anom)
    for i in range(n_anom):
        anom[i, hit[i]] += np.random.choice([-1, 1]) * np.random.uniform(6, 12)
    X = np.vstack([normal, anom]).astype('float32')
    y = np.concatenate([np.zeros(n_norm), np.ones(n_anom)]).astype('int64')
    p = np.random.permutation(len(X))
    return X[p], y[p], [f'sensor_{i}' for i in range(d)]

if CSV_PATH and os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    feats = FEATURES or [c for c in df.select_dtypes('number').columns if c != LABEL_COL]
    X = df[feats].to_numpy('float32')
    y = df[LABEL_COL].to_numpy('int64') if LABEL_COL else np.zeros(len(X), 'int64')
    FEATURES = feats
else:
    print('No CSV_PATH set — synthesizing telemetry.')
    X, y, FEATURES = synthesize()

print('X', X.shape, 'anomaly rate', float(y.mean()), 'features', FEATURES)

In [ ]:
# ---- Split + scale. Fit the scaler on NORMAL rows only (unsupervised training). ----
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y if y.max() > 0 else None, random_state=42)
scaler = StandardScaler().fit(Xtr[ytr == 0] if y.max() > 0 else Xtr)
Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
Xtr_norm = Xtr_s[ytr == 0] if y.max() > 0 else Xtr_s

In [ ]:
# ---- Compact autoencoder (small enough for a pure-Java forward pass) ----
d = X.shape[1]
ae = keras.Sequential([
    keras.layers.Input((d,)),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8,  activation='relu'),
    keras.layers.Dense(4,  activation='relu'),
    keras.layers.Dense(8,  activation='relu'),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(d,  activation='linear'),
])
ae.compile(optimizer='adam', loss='mse')
ae.fit(Xtr_norm, Xtr_norm, epochs=30, batch_size=64, validation_split=0.1, verbose=2)

In [ ]:
# ---- Threshold = 99th percentile of reconstruction error on normal data ----
def mse_err(model, data):
    rec = model.predict(data, verbose=0)
    return np.mean((rec - data) ** 2, axis=1)

err_norm = mse_err(ae, Xtr_norm)
threshold = float(np.percentile(err_norm, 99))
err_te = mse_err(ae, Xte_s)
pred = (err_te > threshold).astype(int)
print('threshold', threshold)
if y.max() > 0:
    print('AUC', round(roc_auc_score(yte, err_te), 4))
    p, r, f, _ = precision_recall_fscore_support(yte, pred, average='binary', zero_division=0)
    print(f'precision {p:.3f} recall {r:.3f} f1 {f:.3f}')

In [ ]:
# ---- Isolation Forest baseline (compare; not exported to Java) ----
iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42).fit(Xtr_norm)
if y.max() > 0:
    print('IsolationForest AUC', round(roc_auc_score(yte, -iso.score_samples(Xte_s)), 4))

In [ ]:
# ---- Export 1: ONNX (optional, for ONNX Runtime Java) ----
try:
    import tf2onnx, onnx
    spec = (tf.TensorSpec((None, d), tf.float32, name='input'),)
    onnx_model, _ = tf2onnx.convert.from_keras(ae, input_signature=spec, opset=13)
    onnx.save(onnx_model, f'{OUT}/iot_autoencoder.onnx')
    print('wrote iot_autoencoder.onnx')
except Exception as e:
    print('ONNX export skipped:', e)

In [ ]:
# ---- Export 2: pure-Java params (scaler + dense weights + threshold) ----
layers = []
for l in ae.layers:
    w = l.get_weights()
    if not w:
        continue
    kernel, bias = w[0], w[1]          # kernel: (in, out)
    act = getattr(l.activation, '__name__', 'linear')
    layers.append({
        'w': kernel.T.tolist(),        # store as (out, in) so Java does W . x
        'b': bias.tolist(),
        'act': 'relu' if act == 'relu' else 'linear',
    })

params = {
    'type': 'autoencoder_mse',
    'features': FEATURES,
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_std':  scaler.scale_.tolist(),
    'threshold':   threshold,
    'layers':      layers,
}
with open(f'{OUT}/iot_anomaly_model.json', 'w') as fh:
    json.dump(params, fh)
print('wrote iot_anomaly_model.json  (', len(layers), 'dense layers )')
print('files in /kaggle/working:', os.listdir(OUT))

## Java integration (pure-Java, no native deps)

Download `iot_anomaly_model.json` from the Kaggle **Output** tab and drop it on each node (path via a `TELEMETRY_MODEL_PATH` env var). Inference per reading:

1. `z[i] = (x[i] - scaler_mean[i]) / scaler_std[i]`
2. forward pass: for each layer `a = act(W·a + b)` (`relu` or `linear`)
3. `error = mean((output - z)^2)`
4. **anomaly if `error > threshold`**

This is ~60 lines of Java (matrix-vector multiply + ReLU). It slots in behind the existing `TelemetryAnomalyDetector` as a second signal, with the current Z-score/ARIMA path as the fallback when no model file is present.